In [2]:
# two apporaches here. 
# first one is an upsampled approach, the best so far
# the second is a bilinear interpolation, not too bad either


In [12]:
## this is different method that uses upsample approach


import numpy as np
from netCDF4 import Dataset
from scipy.interpolate import RegularGridInterpolator
import os

def upsample_parent_terrain(fn_d01="geo_em.d01.nc", fn_d02="geo_em.d02.nc",
                            backup=True):
    """
    Upsample parent-domain terrain (HGT_M) to child-domain resolution
    using bilinear interpolation and overwrite HGT_M in geo_em.d02.nc.
    """

    # === BACKUP geo_em.d02.nc ===
    if backup:
        bak = fn_d02 + ".bak"
        if not os.path.exists(bak):
            print(f"Creating backup: {bak}")
            os.system(f"cp {fn_d02} {bak}")
        else:
            print(f"Backup already exists: {bak}")

    # === OPEN FILES ===
    f1 = Dataset(fn_d01, "r")
    f2 = Dataset(fn_d02, "r+")

    def get2d(nc, varnames):
        """Read 2D variable, removing time dimension if needed"""
        for name in varnames:
            if name in nc.variables:
                v = nc.variables[name][:]
                if v.ndim == 3 and v.shape[0] == 1:
                    return v[0, :, :]
                elif v.ndim == 2:
                    return v
        raise KeyError(f"None of {varnames} found in {nc.filepath()}")

    # === READ VARIABLES ===
    XLAT1  = get2d(f1, ["XLAT_M", "XLAT"])
    XLONG1 = get2d(f1, ["XLONG_M", "XLONG"])
    HGT1   = get2d(f1, ["HGT_M"])

    XLAT2  = get2d(f2, ["XLAT_M", "XLAT"])
    XLONG2 = get2d(f2, ["XLONG_M", "XLONG"])

    ny1, nx1 = XLAT1.shape
    ny2, nx2 = XLAT2.shape
    print(f"Parent grid: {ny1}x{nx1}, Child grid: {ny2}x{nx2}")
    print("Using bilinear interpolation.")

    # === INTERPOLATE USING RegularGridInterpolator ===
    lat1 = XLAT1[:, 0]
    lon1 = XLONG1[0, :]
    interp = RegularGridInterpolator((lat1, lon1), HGT1, method="linear", bounds_error=False, fill_value=None)

    pts_target = np.column_stack((XLAT2.ravel(), XLONG2.ravel()))
    HGT_d01_on_d02 = interp(pts_target).reshape(XLAT2.shape)

    # === WRITE UPSAMPLED HGT INTO geo_em.d02.nc ===
    var = f2.variables["HGT_M"]
    if var.ndim == 3:
        var[0, :, :] = HGT_d01_on_d02
    else:
        var[:, :] = HGT_d01_on_d02
    print(f"Updated HGT_M in {fn_d02}.")

#     # === OPTIONAL: MAP LANDMASK ===
#     if copy_landmask and "LANDMASK" in f1.variables and "LANDMASK" in f2.variables:
#         from scipy.interpolate import griddata
#         LM1 = get2d(f1, ["LANDMASK"])
#         LM_flat = griddata(
#             np.column_stack((XLAT1.ravel(), XLONG1.ravel())),
#             LM1.ravel(),
#             np.column_stack((XLAT2.ravel(), XLONG2.ravel())),
#             method="nearest"
#         )
#         LM2 = LM_flat.reshape(XLAT2.shape)
#         v_lm = f2.variables["LANDMASK"]
#         if v_lm.ndim == 3:
#             v_lm[0, :, :] = LM2
#         else:
#             v_lm[:, :] = LM2
#         print("Mapped LANDMASK from d01 to d02.")

#     # === OPTIONAL: MAP SOILHGT ===
#     if copy_soilhgt:
#         for nm in ("SOILHGT", "SOILHGT_M", "SOILHGT_m"):
#             if nm in f1.variables and nm in f2.variables:
#                 from scipy.interpolate import griddata
#                 S1 = get2d(f1, [nm])
#                 S_flat = griddata(
#                     np.column_stack((XLAT1.ravel(), XLONG1.ravel())),
#                     S1.ravel(),
#                     np.column_stack((XLAT2.ravel(), XLONG2.ravel())),
#                     method="nearest"
#                 )
#                 S2 = S_flat.reshape(XLAT2.shape)
#                 v_s = f2.variables[nm]
#                 if v_s.ndim == 3:
#                     v_s[0, :, :] = S2
#                 else:
#                     v_s[:, :] = S2
#                 print(f"Mapped {nm} from d01 to d02.")
#                 break

    f1.close()
    f2.close()
    print("Done. Re-run real.exe or ndown to regenerate wrfinput_d02.")

# === EXAMPLE USAGE ===
upsample_parent_terrain("geo_em.d01.nc", "geo_em.d02.nc")

Backup already exists: geo_em.d02.nc.bak
Parent grid: 234x258, Child grid: 333x297
Using bilinear interpolation.
Updated HGT_M in geo_em.d02.nc.
Done. Re-run real.exe or ndown to regenerate wrfinput_d02.
